# FE용 데이터셋 제작
1. 고장 개체 비율 fs_sample_train 8 : fs_sample_test 2 (홀드아웃)
2. serial_number 단위에서 failure 비율 유지하여 분할
3. 학습 및 테스트 세트 모두 정상 개체는 고장 개체수의 10배수 샘플링하여 배정
4. 
    - 원본 불균형 1 : 1405.8이지만 타협한 수치
- (같은 serial_number는 train과 test에 동시에 존재하면 안 됨)
- seed = 42

In [1]:
import duckdb
import pandas as pd
import os
from sklearn.model_selection import train_test_split

# ==========================================
# ⚙️ 실행 설정 (원하는 작업만 True로 바꾸세요)
# ==========================================
CONFIG = {
    "BUILD_TRAIN": True,  # 트레인 세트 제작 여부
    "BUILD_TEST": True,    # 테스트 세트 제작 여부
    "TRAIN_RATIO": 10,     # 트레인 정상 샘플링 배수 (1:10)
    "TEST_RATIO": 10,      # 테스트 정상 샘플링 배수 (1:10)
}

# 1. 경로 설정
base_dir = r'../data2/04_feature_engineering'
target_file_name = 'fs_sample_diff.parquet'
target_path = os.path.join(base_dir, target_file_name)
output_train = os.path.join(base_dir, 'fs_sample_train.parquet')
output_test = os.path.join(base_dir, 'fs_sample_test.parquet')

# 2. 파일 필터링
all_files = [f for f in os.listdir(base_dir) if f.endswith('.parquet')]
feature_files = [
    f for f in all_files 
    if not f.startswith('tmp_') 
    and 'train' not in f.lower() 
    and 'test' not in f.lower() 
    and f != target_file_name
]

con = duckdb.connect()
# [메모리 가드] OOM 방지를 위한 안전 설정
con.execute("SET memory_limit = '20GB'")
con.execute("SET threads = 4")

# [Step 0] 청소 및 메타데이터 캐싱
print("🧹 충돌 방지를 위한 파일 청소 및 메타데이터 캐싱 중...")
for f_name, build_flag in [('fs_train.parquet', CONFIG["BUILD_TRAIN"]), ('fs_test.parquet', CONFIG["BUILD_TEST"])]:
    if build_flag and f_name in os.listdir(base_dir):
        try: os.remove(os.path.join(base_dir, f_name))
        except: pass

file_meta = {}
for f in [target_file_name] + feature_files:
    f_path = os.path.join(base_dir, f)
    v_name = f"v_{f.replace('.', '_').replace('-', '_')}"
    con.execute(f"CREATE OR REPLACE VIEW {v_name} AS SELECT * FROM read_parquet('{f_path}')")
    cols = con.execute(f"SELECT * FROM {v_name} WHERE 1=0").df().columns.tolist()
    file_meta[f] = {"view": v_name, "columns": cols}

# [Step 1] SN 개체 단위 8:2 층화 분할 및 정상 개체 10배수 샘플링 (seed=42)
target_view = file_meta[target_file_name]["view"]

# 모든 고장 개체와 정상 개체 목록 분리 (MAX(failure) 기준)
serial_stats = con.execute(f"SELECT serial_number, MAX(failure) as has_failed FROM {target_view} GROUP BY serial_number").df()

failed_serials = serial_stats[serial_stats['has_failed'] == 1]['serial_number'].reset_index(drop=True)
healthy_serials = serial_stats[serial_stats['has_failed'] == 0]['serial_number'].reset_index(drop=True)

# 1. 고장 개체 8:2 홀드아웃 분할 (random_state=42)
train_failed, test_failed = train_test_split(failed_serials, test_size=0.2, random_state=42)

# 2. 정상 개체 8:2 분할 (train/test 간 개체 누수 방지, random_state=42)
train_healthy_pool, test_healthy_pool = train_test_split(healthy_serials, test_size=0.2, random_state=42)

# 3. 각 세트별로 정상 개체를 고장 개체의 10배수 샘플링 (random_state=42)
sampled_train_healthy = train_healthy_pool.sample(n=len(train_failed) * CONFIG["TRAIN_RATIO"], random_state=42)
sampled_test_healthy = test_healthy_pool.sample(n=len(test_failed) * CONFIG["TEST_RATIO"], random_state=42)

# 4. 고장 개체와 샘플링된 정상 개체 병합
train_sn = pd.concat([train_failed, sampled_train_healthy]).reset_index(drop=True)
test_sn = pd.concat([test_failed, sampled_test_healthy]).reset_index(drop=True)

# 5. DuckDB에 임시 등록
con.register('train_sn_list', pd.DataFrame({'serial_number': train_sn}))
con.register('test_sn_list', pd.DataFrame({'serial_number': test_sn}))

def build_query(sn_table, output_file):
    """지정된 개체 리스트에 대해 전체 피처 병합하여 쿼리 빌드"""
    target_cols = file_meta[target_file_name]["columns"]
    seen_columns = set(target_cols)
    
    select_parts = ["s.*"]
    join_parts = []
    
    for i, f in enumerate(feature_files):
        current_cols = file_meta[f]["columns"]
        v_name = file_meta[f]["view"]
        to_exclude = [col for col in current_cols if col in seen_columns]
        exclude_clause = ", ".join([f'"{c}"' for c in to_exclude])
        
        if to_exclude:
            select_parts.append(f'f{i}.* EXCLUDE ({exclude_clause})')
        else:
            select_parts.append(f"f{i}.*")
            
        join_parts.append(f"LEFT JOIN {v_name} f{i} USING (serial_number, date)")
        seen_columns.update(current_cols)
        
    main_sql = f"""
    SELECT {", ".join(select_parts)}
    FROM {target_view} s
    {"".join(join_parts)}
    WHERE s.serial_number IN (SELECT serial_number FROM {sn_table})
    """
    return f"COPY ({main_sql}) TO '{output_file}' (FORMAT 'PARQUET')"

# [Step 2] 실행 (CONFIG 설정에 따라 분기)
if CONFIG["BUILD_TRAIN"]:
    print(f"🏗️ [Train] 제작 시작 (개체 단위 정상 샘플링 1:{CONFIG['TRAIN_RATIO']})")
    con.execute(build_query("train_sn_list", output_train))

if CONFIG["BUILD_TEST"]:
    print(f"🏗️ [Test] 제작 시작 (개체 단위 정상 샘플링 1:{CONFIG['TEST_RATIO']})")
    con.execute(build_query("test_sn_list", output_test))

# [검증]
print("\n⛪ [검증] 최종 데이터 분포 확인")
for name, path, run in [("TRAIN", output_train, CONFIG["BUILD_TRAIN"]), ("TEST", output_test, CONFIG["BUILD_TEST"])]:
    if run:
        stats = con.execute(f"SELECT SUM(CASE WHEN failure=1 THEN 1 ELSE 0 END) as fail, SUM(CASE WHEN failure=0 THEN 1 ELSE 0 END) as healthy FROM read_parquet('{path}')").df()
        print(f"[{name} 행수] 고장: {stats['fail'][0]:,} / 정상: {stats['healthy'][0]:,} (행 비율 1:{round(stats['healthy'][0]/stats['fail'][0], 1)})")
        
        # 개체 수 검증 추가
        drives_stats = con.execute(f"SELECT COUNT(DISTINCT CASE WHEN failure=1 THEN serial_number END) as fail_drives, COUNT(DISTINCT CASE WHEN failure=0 THEN serial_number END) as healthy_drives FROM read_parquet('{path}')").df()
        print(f"[{name} 개체수] 고장: {drives_stats['fail_drives'][0]:,} / 정상: {drives_stats['healthy_drives'][0]:,} (개체 비율 1:{round(drives_stats['healthy_drives'][0]/drives_stats['fail_drives'][0], 1)})")

con.close()
print("\n🚀 설정된 작업이 모두 완료되었습니다!")


🧹 충돌 방지를 위한 파일 청소 및 메타데이터 캐싱 중...
🏗️ [Train] 제작 시작 (비율 1:10)
🏗️ [Test] 제작 시작 (비율 1:100)

⛪ [검증] 최종 데이터 분포 확인
[TRAIN] 고장: 26,858.0 / 정상: 268,580.0 (비율 1:10.0)
[TEST] 고장: 6,715.0 / 정상: 671,500.0 (비율 1:100.0)

🚀 설정된 작업이 모두 완료되었습니다!


In [2]:
import duckdb
import pandas as pd
import os
from sklearn.model_selection import train_test_split

# 1. 필수 경로 재설정
base_dir = r'../data2/04_feature_engineering'
target_file_name = 'fs_sample_diff.parquet'
target_path = os.path.join(base_dir, target_file_name)

# 2. DuckDB 연결
con = duckdb.connect()

print("🔍 데이터 로드 및 8:2 분할 시뮬레이션 중...")

# 3. 전체 시리얼 번호(SN) 목록과 고장 여부 가져오기
serial_stats = con.execute(f"""
    SELECT serial_number, MAX(failure) as has_failed 
    FROM read_parquet('{target_path}') 
    GROUP BY serial_number
""").df()

# 4. 8:2로 분할하여 '테스트 그룹' SN만 추출 (random_state 고정)
_, test_sn = train_test_split(
    serial_stats['serial_number'], 
    test_size=0.2, 
    stratify=serial_stats['has_failed'], 
    random_state=42
)

# 5. 테스트 SN 목록을 DuckDB에 임시 등록
con.register('test_sn_temp', pd.DataFrame({'serial_number': test_sn}))

# 6. 해당 그룹의 모든 행(전수조사)에 대한 불균형 비율 계산
result = con.execute(f"""
    SELECT 
        COUNT(CASE WHEN failure = 1 THEN 1 END) as fail_rows,
        COUNT(CASE WHEN failure = 0 THEN 1 END) as healthy_rows,
        ROUND(COUNT(CASE WHEN failure = 0 THEN 1 END) / NULLIF(COUNT(CASE WHEN failure = 1 THEN 1 END), 0), 2) as imbalance_ratio
    FROM read_parquet('{target_path}')
    WHERE serial_number IN (SELECT serial_number FROM test_sn_temp)
""").df()

print("\n📊 [테스트 SN 그룹(전수조사) 불균형 결과]")
print("-" * 40)
print(result.to_string(index=False))
print("-" * 40)
print(f"💡 결론: 테스트 개체들을 전수 조사하면 1:{result['imbalance_ratio'][0]} 비율이 나옵니다.")

con.close()


🔍 데이터 로드 및 8:2 분할 시뮬레이션 중...

📊 [테스트 SN 그룹(전수조사) 불균형 결과]
----------------------------------------
 fail_rows  healthy_rows  imbalance_ratio
      6744       9513579          1410.67
----------------------------------------
💡 결론: 테스트 개체들을 전수 조사하면 1:1410.67 비율이 나옵니다.


## 3. RFE 샘플링 무결성 검증 테스트 (Verification Tests)

생성된 훈련용(Train) 및 테스트용(Test) 샘플링 데이터셋의 파일 존재 여부, 두 데이터셋 간 개체(serial_number) 중복 누수 차단 여부, 그리고 고장 개체 대비 정상 개체의 10배수 샘플링 비율을 수학적으로 엄밀히 입증합니다.

In [ ]:
import duckdb
import os

con = duckdb.connect()

output_train = r"../data2/04_feature_engineering/fs_sample_train.parquet"
output_test = r"../data2/04_feature_engineering/fs_sample_test.parquet"

print("🔍 [5-A단계 무결성 검증] 시작...")

try:
    # 1. 파일 존재 여부 검증
    print("Test 1: 출력 파일 존재 여부 검증")
    assert os.path.exists(output_train), f"오류: {output_train} 파일이 생성되지 않았습니다."
    assert os.path.exists(output_test), f"오류: {output_test} 파일이 생성되지 않았습니다."
    print("  -> [PASS] 출력 파일 존재 확인.")

    # 2. Train / Test 간 개체 중복 누수(Leakage) 차단 검증
    print("Test 2: Train과 Test 간 개체(serial_number) 중복 누수 검증")
    overlap_count = con.execute(f"""
        SELECT COUNT(DISTINCT tr.serial_number)
        FROM read_parquet('{output_train}') tr
        JOIN read_parquet('{output_test}') te ON tr.serial_number = te.serial_number
    """).fetchone()[0]
    assert overlap_count == 0, f"오류: Train과 Test 세트 사이에 중복되는 serial_number가 {overlap_count}개 존재합니다!"
    print("  -> [PASS] Train/Test 간 개체 누수 없음.")

    # 3. 개체 비율 1:10 검증
    print("Test 3: 고장 개체와 정상 개체의 10배수 샘플링 비율 검증")
    for name, path in [("TRAIN", output_train), ("TEST", output_test)]:
        drives_stats = con.execute(f"""
            SELECT 
                COUNT(DISTINCT CASE WHEN failure=1 THEN serial_number END) as fail_drives, 
                COUNT(DISTINCT CASE WHEN failure=0 THEN serial_number END) as healthy_drives 
            FROM read_parquet('{path}')
        """).fetchone()
        fail_drives, healthy_drives = drives_stats[0], drives_stats[1]
        
        expected_healthy = fail_drives * 10
        assert healthy_drives == expected_healthy, f"오류: {name} 세트의 정상 개체수({healthy_drives})가 고장 개체수({fail_drives})의 10배({expected_healthy})가 아닙니다!"
    print("  -> [PASS] 고장 개체 대비 정상 개체 10배수 샘플링 정합성 확인.")

    # 4. 결측치 유무 검증
    print("Test 4: 필수 컬럼 결측치 존재 여부 검증")
    for name, path in [("TRAIN", output_train), ("TEST", output_test)]:
        null_counts = con.execute(f"""
            SELECT 
                COUNT(*) - COUNT(serial_number) as null_sn,
                COUNT(*) - COUNT(date) as null_dt,
                COUNT(*) - COUNT(failure) as null_fl
            FROM read_parquet('{path}')
        """).fetchone()
        assert sum(null_counts) == 0, f"오류: {name} 세트에 결측치가 존재합니다: {null_counts}"
    print("  -> [PASS] 필수 메타 컬럼 결측치 없음.")

    print("\n🏆 [5-A단계 정합성 검증 완료] 모든 엄격한 테스트 조건을 만족합니다!")
finally:
    con.close()
